# NFL Upset Prediction: Feature Engineering

This notebook runs the feature engineering pipeline:
1. Rolling average features (offense/defense stats)
2. Matchup differential features
3. Target variable calculation
4. Feature distribution analysis
5. Data leakage verification

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project imports
import sys
sys.path.insert(0, '..')

from src.data.nfl_loader import load_schedules
from src.data.betting_loader import load_betting_data
from src.data.merger import merge_nfl_betting_data
from src.features.pipeline import FeaturePipeline
from src.features.target import create_upset_target
from src.features.rolling import calculate_rolling_stats, ROLLING_COLUMNS

# Display settings
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load and Merge Data

In [ ]:
# Load data
seasons = list(range(2015, 2024))
nfl_data = load_schedules(seasons=seasons)
betting_data = load_betting_data(min_season=2015, max_season=2023)

# Merge
merged_data, audit = merge_nfl_betting_data(nfl_data, betting_data)
print(f"Starting with {len(merged_data):,} merged records")
print(f"Merge rate: {audit.merge_rate:.1%}")

## 2. Create Target Variable

In [ ]:
# Create upset target
data_with_target = create_upset_target(merged_data)

print(f"Records with valid target: {data_with_target['is_upset'].notna().sum():,}")
print(f"Records excluded: {data_with_target['is_upset'].isna().sum():,}")

# Show target distribution
valid = data_with_target.dropna(subset=['is_upset'])
print(f"\nTarget distribution:")
print(valid['is_upset'].value_counts())
print(f"\nUpset rate: {valid['is_upset'].mean():.1%}")

## 3. Run Feature Pipeline

In [ ]:
# Initialize and run pipeline
pipeline = FeaturePipeline()
features_df = pipeline.fit_transform(merged_data)

print(f"Final dataset: {len(features_df):,} records")
print(f"Total features: {len(features_df.columns)}")
print(f"\nFeature columns:")
print(sorted(features_df.columns.tolist()))

In [ ]:
# Show sample of engineered features
feature_cols = [c for c in features_df.columns if c.startswith(('rolling_', 'matchup_'))]
features_df[['season', 'week', 'home_team', 'away_team', 'is_upset'] + feature_cols[:10]].head(10)

## 4. Feature Distribution Analysis

In [ ]:
# Feature statistics
feature_cols = [c for c in features_df.columns if c.startswith(('rolling_', 'matchup_', 'spread'))]
feature_stats = features_df[feature_cols].describe().T
feature_stats['missing_pct'] = features_df[feature_cols].isna().mean() * 100
feature_stats.sort_values('missing_pct', ascending=False).head(20)

In [ ]:
# Plot distributions of key features
key_features = ['spread_magnitude', 'matchup_offense_diff', 'matchup_defense_diff']
key_features = [f for f in key_features if f in features_df.columns]

if key_features:
    fig, axes = plt.subplots(1, len(key_features), figsize=(5*len(key_features), 4))
    if len(key_features) == 1:
        axes = [axes]
    
    for ax, col in zip(axes, key_features):
        data = features_df[col].dropna()
        ax.hist(data, bins=30, edgecolor='black', alpha=0.7)
        ax.set_xlabel(col)
        ax.set_ylabel('Count')
        ax.set_title(f'{col}\nMean: {data.mean():.2f}, Std: {data.std():.2f}')
    
    plt.tight_layout()
    plt.savefig('../results/figures/feature_distributions.png', dpi=150)
    plt.show()

In [ ]:
# Feature correlations with target
valid_features = features_df.dropna(subset=['is_upset'])
correlations = valid_features[feature_cols].corrwith(valid_features['is_upset']).sort_values(ascending=False)

print("Top features correlated with upsets:")
print(correlations.head(10))
print("\nBottom features (negatively correlated):")
print(correlations.tail(10))

## 5. Data Leakage Verification

In [ ]:
# Verify rolling features use only past data
# Check: for each game, rolling stats should NOT include current game

# Pick a random team and season
test_team = 'KC'
test_season = 2022

team_games = features_df[
    ((features_df['home_team'] == test_team) | (features_df['away_team'] == test_team)) &
    (features_df['season'] == test_season)
].sort_values('week')

print(f"Verifying rolling stats for {test_team} in {test_season}:")
print(f"Games found: {len(team_games)}")

# Week 1 should have NaN rolling stats (no prior games)
week1 = team_games[team_games['week'] == 1]
if len(week1) > 0:
    rolling_cols = [c for c in team_games.columns if c.startswith('rolling_')]
    if rolling_cols:
        week1_rolling = week1[rolling_cols].iloc[0]
        all_na = week1_rolling.isna().all()
        print(f"\nWeek 1 rolling stats all NaN: {all_na}")
        if not all_na:
            print("WARNING: Week 1 has non-NaN rolling stats - possible data leakage!")
    else:
        print("No rolling columns found in output")

In [ ]:
# Check that rolling window respects temporal order
# Week N should only use data from weeks 1 to N-1

print("\nSample rolling stats progression for", test_team, test_season, ":")
cols_to_show = ['week', 'home_team', 'away_team', 'home_score', 'away_score']
rolling_sample = [c for c in team_games.columns if 'rolling' in c][:3]
team_games[cols_to_show + rolling_sample].head(8)

## 6. Save Processed Dataset

In [ ]:
# Remove rows with missing target
final_dataset = features_df.dropna(subset=['is_upset']).copy()
print(f"Final dataset size: {len(final_dataset):,}")

# Identify feature columns for modeling
feature_columns = [c for c in final_dataset.columns 
                   if c.startswith(('rolling_', 'matchup_', 'spread'))]
print(f"Feature columns for modeling: {len(feature_columns)}")

# Save
final_dataset.to_parquet('../data/processed/features_engineered.parquet', index=False)
print(f"\nSaved to data/processed/features_engineered.parquet")

# Save feature column list
with open('../data/processed/feature_columns.txt', 'w') as f:
    f.write('\n'.join(feature_columns))
print(f"Feature list saved to data/processed/feature_columns.txt")

In [ ]:
# Final summary
print("=" * 50)
print("FEATURE ENGINEERING SUMMARY")
print("=" * 50)
print(f"\nTotal games with features: {len(final_dataset):,}")
print(f"Season range: {final_dataset['season'].min()} - {final_dataset['season'].max()}")
print(f"Number of features: {len(feature_columns)}")
print(f"Target upset rate: {final_dataset['is_upset'].mean():.1%}")
print(f"\nFeature categories:")
print(f"  - Rolling averages: {len([c for c in feature_columns if c.startswith('rolling_')])}")
print(f"  - Matchup diffs: {len([c for c in feature_columns if c.startswith('matchup_')])}")
print(f"  - Spread features: {len([c for c in feature_columns if c.startswith('spread')])}")